# Zlab Cloud · Five-second stop validation
Use a CPU session, enable Internet, and run all cells. Compare checkpoint 999 and both versions of 1199 under the same five-second hold at 2.8 m/s.


In [ ]:
import os
from pathlib import Path
import subprocess
import tempfile

repo = Path(tempfile.mkdtemp(prefix="zlab-cloud-")) / "Zlab-cloud"
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", "main",
    "https://github.com/hafizadha/Zlab-cloud.git", str(repo),
], check=True)
project = repo / "Zlab_Gym"
launcher = project / "train_ball_stability.sh"
if not launcher.is_file():
    raise RuntimeError("This checkout does not contain the ball stability launcher.")
subprocess.run(["git", "-C", str(repo), "log", "-1", "--oneline"], check=True)


import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], cwd=project, check=True)

In [ ]:
WORK_DIR = Path("/kaggle/working/zlab-robocup-five-second-validation")
FOOT_FRICTION_RANGE = (0.35, 1.15)
candidate = project / "checkpoints/model_1199_five_second.pt"
alternative = project / "checkpoints/model_1199.pt"
control = project / "checkpoints/model_999.pt"
subprocess.run([
    sys.executable, "scripts/train_stability_blocks.py", str(candidate),
    "--work-dir", str(WORK_DIR), "--validate-against", str(control),
    "--validation-alternative", str(alternative),
], cwd=project, env=dict(
    os.environ,
    ZLAB_FOOT_FRICTION_MIN=str(FOOT_FRICTION_RANGE[0]),
    ZLAB_FOOT_FRICTION_MAX=str(FOOT_FRICTION_RANGE[1]),
), check=True)

In [ ]:
import json

report = json.loads((WORK_DIR / "validation.json").read_text())
print("Selected checkpoint:", Path(report["selected_checkpoint"]).name)
print("Selected weights:", WORK_DIR / "selected.pt")
print(json.dumps(report, indent=2))